In [ ]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased", cache_dir='/data/user_data/jiaruil5/.cache/')
model = BertForMaskedLM.from_pretrained("google-bert/bert-base-multilingual-cased", cache_dir='/data/user_data/jiaruil5/.cache/', output_attentions=True)

In [15]:
# Construct a generic context with masks around "fox"
input_text = "我{start_loc}提出了一个设计对称元素集的深度网络的原则:"

for i in range(3):
    reformatted_input_text = input_text.format(start_loc="[MASK]" * i)
    
    inputs = tokenizer(reformatted_input_text, return_tensors="pt")
    print(inputs)

    # Forward pass to get predictions and attention weights
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = outputs.logits
        attentions = outputs.attentions

    # Get predicted tokens for [MASK] positions
    masked_indices = [i for i, token_id in enumerate(inputs['input_ids'][0]) if tokenizer.decode([token_id]) == "[MASK]"]

    # Retrieve the top predicted tokens for each [MASK]
    top_k = 10  # Number of top predictions to consider
    predicted_tokens = {}
    for idx in masked_indices:
        probs = torch.softmax(predictions[0, idx], dim=-1)
        top_tokens = torch.topk(probs, top_k).indices
        predicted_tokens[f"Position {idx}"] = [tokenizer.decode([token]).strip() for token in top_tokens]

    print("Top predicted tokens for positions:", predicted_tokens)


{'input_ids': tensor([[ 101, 3976, 4181, 2527, 2146, 2072, 2102, 7318, 7298, 3442, 5964, 2426,
         6195, 8272, 5718, 5052, 3670, 6397, 6345, 5718, 2715, 2544,  131,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Top predicted tokens for positions: {}
{'input_ids': tensor([[ 101, 3976,  103, 4181, 2527, 2146, 2072, 2102, 7318, 7298, 3442, 5964,
         2426, 6195, 8272, 5718, 5052, 3670, 6397, 6345, 5718, 2715, 2544,  131,
          102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]])}
Top predicted tokens for positions: {'Position 2': ['们', '要', '了', '是', '里', '我', '的', '国', '式', '还']}
{'input_ids': tensor([[ 101, 3976,  103,  103, 4181, 

In [17]:
model.eval()  # Set model to evaluation mode

# Example input (first n tokens and last n+i+j tokens)
first_n_tokens = "我"  # Example first part
last_tokens = "提出了一个设计对称元素集的深度网络的原则:"  # Example last part

# Tokenize the input parts
first_n_ids = tokenizer.encode(first_n_tokens, add_special_tokens=False)
last_ids = tokenizer.encode(last_tokens, add_special_tokens=False)

# Assume the gap is fully masked initially (length unknown, start with a large number of masks)
max_possible_gap = 3  # You can choose a reasonable upper bound
input_ids = first_n_ids + [tokenizer.mask_token_id] * max_possible_gap + last_ids

# Convert input_ids to a PyTorch tensor and add batch dimension
input_ids = torch.tensor([input_ids])

# Iteratively predict masked tokens
with torch.no_grad():
    for _ in range(max_possible_gap):  # Limit the number of iterations to avoid infinite loops
        # Forward pass
        outputs = model(input_ids)
        predictions = outputs.logits
        print(predictions.shape)

        # Find the first masked position
        mask_positions = [i for i, token_id in enumerate(input_ids[0].tolist()) if token_id == tokenizer.mask_token_id]
        if not mask_positions:
            break  # No more masks, stop the loop

        # Predict the token for the first masked position
        first_mask_position = mask_positions[0]
        predicted_index = torch.argmax(predictions[0, first_mask_position]).item()
        predicted_token = tokenizer.decode([predicted_index])

        # Replace the mask with the predicted token
        input_ids[0, first_mask_position] = predicted_index

# Decode the final output
final_output = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("Predicted sequence:", final_output)


torch.Size([1, 25, 119547])
torch.Size([1, 25, 119547])
torch.Size([1, 25, 119547])
Predicted sequence: 我 在 这 里 提 出 了 一 个 设 计 对 称 元 素 集 的 深 度 网 络 的 原 则 :


In [21]:
import spacy
# import pymorphy2
# from camel_tools.morphology.analyzer import Analyzer as CamelAnalyzer
# import jieba
# import janome.tokenizer

# Load English and French spaCy models
nlp_en = spacy.blank("en")
nlp_fr = spacy.blank("fr")

# # Load Russian morphological analyzer
# morph_russian = pymorphy2.MorphAnalyzer()

# # Initialize Arabic analyzer (using camel_tools)
# camel_analyzer = CamelAnalyzer.pretrained()

# # Initialize Japanese tokenizer
# janome_tokenizer = janome.tokenizer.Tokenizer()

def generate_transformations(text, lang='en'):
    transformations = {}
    
    # English and French transformations using spaCy
    if lang in ['en', 'fr']:
        doc = nlp_en(text) if lang == 'en' else nlp_fr(text)
        transformations['lemma'] = [token.lemma_ for token in doc]
        transformations['lowercase'] = [token.lower_ for token in doc]
        transformations['uppercase'] = [token.text.upper() for token in doc]
        transformations['plural'] = [
            token.text + 's' if not token.text.endswith('s') else token.text for token in doc
        ]
        transformations['possessive'] = [token.text + "'s" for token in doc]
        transformations['derivations'] = [token.text + 'ness' if lang == 'en' else token.text + 'ité' for token in doc]
    
    # # Russian transformations using pymorphy2
    # elif lang == 'ru':
    #     words = text.split()
    #     transformations['inflections'] = [
    #         [parse.inflect({'plur', 'nomn'}).word for parse in morph_russian.parse(word) if parse.inflect({'plur', 'nomn'})]
    #         for word in words
    #     ]
    
    # # Arabic transformations using CAMeL Tools
    # elif lang == 'ar':
    #     analyses = camel_analyzer.analyze(text)
    #     transformations['analysis'] = [analysis['diac'] for analysis in analyses]
    
    # # Chinese transformations using jieba
    # elif lang == 'zh':
    #     tokens = list(jieba.cut(text))
    #     transformations['tokens'] = tokens
    
    # # Japanese transformations using Janome
    # elif lang == 'ja':
    #     tokens = [token.surface for token in janome_tokenizer.tokenize(text)]
    #     transformations['tokens'] = tokens
    
    else:
        transformations['error'] = "Unsupported language."
    
    return transformations

# Example usage
results = {
    "English": generate_transformations("cat", lang='en'),
    "French": generate_transformations("chat", lang='fr'),
    # "Russian": generate_transformations("кошка", lang='ru'),
    # "Arabic": generate_transformations("قطة", lang='ar'),
    # "Chinese": generate_transformations("猫", lang='zh'),
    # "Japanese": generate_transformations("猫", lang='ja')
}

results



{'English': {'lemma': [''],
  'lowercase': ['cat'],
  'uppercase': ['CAT'],
  'plural': ['cats'],
  'possessive': ["cat's"],
  'derivations': ['catness']},
 'French': {'lemma': [''],
  'lowercase': ['chat'],
  'uppercase': ['CHAT'],
  'plural': ['chats'],
  'possessive': ["chat's"],
  'derivations': ['chatité']}}

In [23]:
import gensim.downloader as api

# Load a pretrained model (Word2Vec)
model = api.load('word2vec-google-news-300')  # This downloads the Google News pretrained Word2Vec model (3 million words)

# Find similar words
similar_words = model.most_similar('cat', topn=5)  # Top 5 most similar words to 'cat'
print(similar_words)


TypeError: 